In [1]:
# Validation qualité dans le notebook

import great_expectations as ge
import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("HospitalOptimizer-Quality") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Charger Gold en Pandas pour Great Expectations
df_spark = spark.read.parquet("/home/jovyan/data/gold/mimic_full_features.parquet")
df = df_spark.toPandas()

print(f"✅ Données chargées : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

/usr/local/spark/python/pyspark/sql/pandas/utils.py:37: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pandas.__version__) < LooseVersion(minimum_pandas_version):
/usr/local/spark/python/pyspark/sql/pandas/utils.py:37: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pandas.__version__) < LooseVersion(minimum_pandas_version):
/usr/local/spark/python/pyspark/sql/pandas/utils.py:37: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pandas.__version__) < LooseVersion(minimum_pandas_version):
/usr/local/spark/python/pyspark/sql/pandas/utils.py:37: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pandas.__version__) < LooseVersion(minimum_pandas_version):


✅ Données chargées : 84,886 lignes × 31 colonnes


In [5]:
# Définir les règles de qualité

import pandas as pd

# ── Validateur maison ─────────────────────────────────────────────
def check(name, condition, df):
    n_fail = (~condition).sum()
    pct    = n_fail / len(df) * 100
    status = "✅ OK" if n_fail == 0 else "❌ ÉCHEC"
    print(f"{status} | {name:<40} | {n_fail:>6} anomalies ({pct:.2f}%)")
    return n_fail == 0

print("=" * 65)
print("RAPPORT QUALITÉ DES DONNÉES")
print("=" * 65)

results = []
results.append(check("LOSdays entre 0 et 200",         df["LOSdays"].between(0, 200),          df))
results.append(check("gender = M ou F",                df["gender"].isin(["M", "F"]),           df))
results.append(check("hadm_id jamais null",            df["hadm_id"].notnull(),                 df))
results.append(check("num_labs >= 0",                  df["num_labs"] >= 0,                     df))
results.append(check("abnormal_lab_rate entre 0 et 1", df["abnormal_lab_rate"].between(0, 1),   df))
results.append(check("high_risk = 0 ou 1",             df["high_risk"].isin([0, 1]),             df))
results.append(check("complexity_score >= 0",          df["complexity_score"] >= 0,             df))
results.append(check("admission_type jamais null",     df["admission_type"].notnull(),           df))

print("=" * 65)
passed = sum(results)
print(f"Résultat : {passed}/{len(results)} règles validées")

RAPPORT QUALITÉ DES DONNÉES
✅ OK | LOSdays entre 0 et 200                   |      0 anomalies (0.00%)
✅ OK | gender = M ou F                          |      0 anomalies (0.00%)
✅ OK | hadm_id jamais null                      |      0 anomalies (0.00%)
✅ OK | num_labs >= 0                            |      0 anomalies (0.00%)
✅ OK | abnormal_lab_rate entre 0 et 1           |      0 anomalies (0.00%)
✅ OK | high_risk = 0 ou 1                       |      0 anomalies (0.00%)
✅ OK | complexity_score >= 0                    |      0 anomalies (0.00%)
✅ OK | admission_type jamais null               |      0 anomalies (0.00%)
Résultat : 8/8 règles validées


In [6]:
# Alerting automatique sur anomalies
# On va créer une fonction qui envoie une alerte si une règle est violée.

import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("DataQuality")

def validate_and_alert(df, rules):
    """
    Valide les règles et alerte si anomalie détectée.
    rules = liste de tuples (nom, condition)
    """
    failed_rules = []

    for name, condition in rules:
        n_fail = (~condition).sum()
        pct    = n_fail / len(df) * 100

        if n_fail > 0:
            failed_rules.append((name, n_fail, pct))
            logger.warning(f"⚠️  ANOMALIE : {name} — {n_fail} lignes ({pct:.2f}%)")
        else:
            logger.info(f"✅ OK : {name}")

    # Résumé
    if failed_rules:
        logger.error(f"❌ {len(failed_rules)} règle(s) en échec — action requise !")
    else:
        logger.info(f"✅ Toutes les règles validées — données prêtes pour les modèles")

    return len(failed_rules) == 0

# ── Définir les règles ────────────────────────────────────────────
rules = [
    ("LOSdays entre 0 et 200",          df["LOSdays"].between(0, 200)),
    ("gender = M ou F",                 df["gender"].isin(["M", "F"])),
    ("hadm_id jamais null",             df["hadm_id"].notnull()),
    ("num_labs >= 0",                   df["num_labs"] >= 0),
    ("abnormal_lab_rate entre 0 et 1",  df["abnormal_lab_rate"].between(0, 1)),
    ("high_risk = 0 ou 1",              df["high_risk"].isin([0, 1])),
]

# ── Lancer la validation ──────────────────────────────────────────
is_valid = validate_and_alert(df, rules)

INFO:DataQuality:✅ OK : LOSdays entre 0 et 200
INFO:DataQuality:✅ OK : gender = M ou F
INFO:DataQuality:✅ OK : hadm_id jamais null
INFO:DataQuality:✅ OK : num_labs >= 0
INFO:DataQuality:✅ OK : abnormal_lab_rate entre 0 et 1
INFO:DataQuality:✅ OK : high_risk = 0 ou 1
INFO:DataQuality:✅ Toutes les règles validées — données prêtes pour les modèles
